# Gloss → English sentence model (ASL Translator, step 2)

Fine-tunes a seq2seq model (default **`google/flan-t5-base`**) that turns the sign
sequence detected in a video into an English sentence, e.g.
`X-YOUR NAME WHAT` → *"What is your name?"*.

Training runs in two stages:

1. **ASLG-PC12** (≈81k unique gloss/English pairs) teaches general gloss → English
   grammar. It is synthetic and formal (European Parliament text).
2. **In-domain pairs** built from the app's 22 signs (`training/data/indomain_train.jsonl`)
   adapt the model to everyday phrases, mixed with a small ASLG "replay" sample so it
   doesn't forget stage 1.

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
At the end a `gloss2text.zip` is downloaded; see the last cell for how to install it.

The text conventions (gloss spelling, normalisation) come from
`backend/app/services/gloss_text.py`, the same file the backend uses, so training and
inference always match.

In [ ]:
# 1. Install dependencies (transformers pinned to the backend's version)
!pip install -q "transformers==4.57.6" "sacrebleu==2.5.1" accelerate sentencepiece

## 2. Get the project files

Clones the `ASL` branch (push your latest commits first). If the repository is private,
either clone with a token (`https://<token>@github.com/...`) or upload a zip of the repo
and set `REPO_DIR` to the extracted folder.

In [ ]:
import os, sys

REPO_URL = "https://github.com/duwcston/VSL-Detection.git"
REPO_BRANCH = "ASL"
REPO_DIR = "/content/VSL-Detection"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}

sys.path.insert(0, f"{REPO_DIR}/backend")
from app.services.gloss_text import LABEL_TO_GLOSS, clean_gloss, labels_to_gloss, normalize_text, prettify

DATA_DIR = f"{REPO_DIR}/training/data"
print(sorted(os.listdir(DATA_DIR)))

## 3. Configuration

For the thesis ablations, change one setting at a time and note the dev scores printed
after each stage (e.g. `flan-t5-small` vs `flan-t5-base` vs `bart-base`, or
`RUN_STAGE1 = False` to measure what ASLG-PC12 contributes).

In [ ]:
MODEL_NAME = "google/flan-t5-base"  # or "google/flan-t5-small", "facebook/bart-base"

RUN_STAGE1 = True   # ASLG-PC12 pre-training
RUN_STAGE2 = True   # in-domain fine-tuning on the 22 signs

IS_T5 = "t5" in MODEL_NAME.lower()
STAGE1_EPOCHS = 1
STAGE2_EPOCHS = 8
STAGE1_LR = 3e-4 if IS_T5 else 5e-5
STAGE2_LR = 1e-4 if IS_T5 else 3e-5
BATCH_SIZE = 32
MAX_SOURCE_LEN = 64
MAX_TARGET_LEN = 64
ASLG_REPLAY = 3000      # ASLG pairs mixed into stage 2 so stage 1 isn't forgotten
NUM_BEAMS = 4           # same as SENTENCE_NUM_BEAMS in the backend config

QUICK_TEST = False      # True: tiny subsets, just to check the notebook runs end to end
OUTPUT_DIR = "/content/gloss2text"
RUNS_DIR = "/content/runs"
SEED = 42

## 4. Load the data

In [ ]:
import json, random
import pandas as pd
from huggingface_hub import hf_hub_download

def read_jsonl(name):
    with open(f"{DATA_DIR}/{name}", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

# ASLG-PC12: strip the BOM/newlines, drop punctuation from glosses (detected signs
# never contain any) and normalise the English side.
parquet = hf_hub_download("achrafothman/aslg_pc12", "data/train-00000-of-00001.parquet", repo_type="dataset")
aslg = pd.read_parquet(parquet)
aslg["gloss"] = aslg["gloss"].map(clean_gloss)
aslg["text"] = aslg["text"].map(normalize_text)
aslg = aslg[(aslg.gloss != "") & (aslg.text != "")].drop_duplicates()
aslg = aslg.sample(frac=1, random_state=SEED).reset_index(drop=True)
aslg_val, aslg_train = aslg[:1000], aslg[1000:]

# In-domain pairs (from build_dataset.py). Glosses are recomputed from the labels so
# they always follow the current LABEL_TO_GLOSS mapping.
indomain = read_jsonl("indomain_train.jsonl")
for row in indomain:
    row["gloss"] = labels_to_gloss(row["labels"])
dev = read_jsonl("dev.jsonl")

if QUICK_TEST:
    aslg_train, aslg_val, indomain = aslg_train[:400], aslg_val[:50], indomain[:200]
    STAGE1_EPOCHS, STAGE2_EPOCHS, ASLG_REPLAY = 1, 1, 100

print(f"ASLG-PC12: {len(aslg_train)} train / {len(aslg_val)} val")
print(f"in-domain: {len(indomain)}   dev: {len(dev)}")
print(aslg_train.head(3).to_string())

## 5. Model, datasets and helpers

In [ ]:
import torch, sacrebleu
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments, set_seed)

set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)


class PairDataset(torch.utils.data.Dataset):
    def __init__(self, sources, targets):
        self.inputs = tokenizer(list(sources), max_length=MAX_SOURCE_LEN, truncation=True)
        self.labels = tokenizer(text_target=list(targets), max_length=MAX_TARGET_LEN, truncation=True)["input_ids"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {"input_ids": self.inputs["input_ids"][i],
                "attention_mask": self.inputs["attention_mask"][i],
                "labels": self.labels[i]}


def make_trainer(train_ds, val_ds, epochs, lr, run_name):
    bf16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
    # T5 overflows in fp16, so on a T4 (no bf16) it trains in fp32; BART is fine in fp16.
    fp16 = DEVICE == "cuda" and not bf16 and not IS_T5
    args = Seq2SeqTrainingArguments(
        output_dir=f"{RUNS_DIR}/{run_name}",
        num_train_epochs=epochs,
        learning_rate=lr,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        warmup_ratio=0.05,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        logging_steps=50,
        bf16=bf16,
        fp16=fp16,
        report_to="none",
        seed=SEED,
    )
    return Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                          data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
                          processing_class=tokenizer)


def generate(glosses, batch_size=32):
    model.eval()
    outputs = []
    for i in range(0, len(glosses), batch_size):
        batch = tokenizer(glosses[i:i + batch_size], return_tensors="pt", padding=True,
                          truncation=True, max_length=MAX_SOURCE_LEN).to(model.device)
        with torch.inference_mode():
            ids = model.generate(**batch, num_beams=NUM_BEAMS, max_new_tokens=32)
        outputs += tokenizer.batch_decode(ids, skip_special_tokens=True)
    return outputs


def evaluate_dev(title, show=8):
    # Same metrics as training/evaluate.py (BLEU/chrF on normalised text, multi-reference).
    hyps = [normalize_text(h) for h in generate([labels_to_gloss(p["labels"]) for p in dev])]
    refs = [[normalize_text(r) for r in p["references"]] for p in dev]
    n = max(len(r) for r in refs)
    streams = [[r[i] if i < len(r) else r[0] for r in refs] for i in range(n)]
    scores = {"BLEU": round(sacrebleu.corpus_bleu(hyps, streams).score, 2),
              "chrF": round(sacrebleu.corpus_chrf(hyps, streams).score, 2)}
    print(f"\n== {title}: {scores}")
    for pair, hyp in list(zip(dev, hyps))[:show]:
        print(f"  {', '.join(pair['labels']):<32} -> {prettify(hyp)}")
    return scores


results = {"before training": evaluate_dev("Before training (zero-shot)")}

## 6. Stage 1: ASLG-PC12

In [ ]:
if RUN_STAGE1:
    trainer = make_trainer(PairDataset(aslg_train.gloss, aslg_train.text),
                           PairDataset(aslg_val.gloss, aslg_val.text),
                           STAGE1_EPOCHS, STAGE1_LR, "stage1")
    trainer.train()
    model = trainer.model
    results["after stage 1"] = evaluate_dev("After stage 1 (ASLG-PC12)")

## 7. Stage 2: in-domain phrases (+ ASLG replay)

In [ ]:
if RUN_STAGE2:
    rows = indomain[:]
    random.Random(SEED).shuffle(rows)
    n_val = max(1, len(rows) // 10)
    val_rows, train_rows = rows[:n_val], rows[n_val:]
    replay = aslg_train.sample(n=min(ASLG_REPLAY, len(aslg_train)), random_state=SEED)

    sources = [r["gloss"] for r in train_rows] + list(replay.gloss)
    targets = [r["text"] for r in train_rows] + list(replay.text)
    trainer = make_trainer(PairDataset(sources, targets),
                           PairDataset([r["gloss"] for r in val_rows], [r["text"] for r in val_rows]),
                           STAGE2_EPOCHS, STAGE2_LR, "stage2")
    trainer.train()
    model = trainer.model
    results["after stage 2"] = evaluate_dev("After stage 2 (in-domain)", show=len(dev))

print(json.dumps(results, indent=2))

## 8. Try it

In [ ]:
examples = [["Your", "Name", "What"], ["I", "Thirsty"], ["Help", "Doctor"],
            ["Hello", "How", "You"], ["You", "Eat", "What"], ["Thank you", "Doctor"]]
for labels, output in zip(examples, generate([labels_to_gloss(l) for l in examples])):
    print(f"{', '.join(labels):<28} -> {prettify(output)}")

## 9. Save and download

Downloads `gloss2text.zip`. To use it in the app:

1. Extract it so the files land in `backend/models/gloss2text/` (`config.json`,
   `model.safetensors`, tokenizer files, `training_info.json`).
2. Run `python training/evaluate.py --noisy --show 10` for the final test-set scores.
3. Restart the backend; uploaded videos now get a *Translation* under the sign sequence.

In [ ]:
import shutil

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
info = {
    "base_model": MODEL_NAME,
    "stages": {"aslg_pc12": RUN_STAGE1, "in_domain": RUN_STAGE2},
    "hyperparameters": {"stage1_epochs": STAGE1_EPOCHS, "stage2_epochs": STAGE2_EPOCHS,
                        "stage1_lr": STAGE1_LR, "stage2_lr": STAGE2_LR, "batch_size": BATCH_SIZE,
                        "aslg_replay": ASLG_REPLAY, "seed": SEED, "quick_test": QUICK_TEST},
    "label_to_gloss": LABEL_TO_GLOSS,
    "dev_results": results,
}
with open(f"{OUTPUT_DIR}/training_info.json", "w") as f:
    json.dump(info, f, indent=2)

archive = shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
print("Saved", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass